# 06. U10 Plate Dataset V2

B00 중심 원판 학습 데이터를 서로 다른 사람과 배경으로 확장합니다. 이 노트북은 **후보 확인 → 6개 영상 선택 → 120프레임 추출 → 수동 bbox → contact sheet 검수 → dataset YAML**까지만 담당합니다. 모델 학습과 A/B/C 평가는 다음 checkpoint에서 추가합니다.

In [ ]:
from pathlib import Path
import math
import os
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from snatch_analysis.config import B00_VIDEO_ID
from snatch_analysis.metadata import (
    save_video_manifest,
    set_has_plate_labels,
)
from snatch_analysis.plate import (
    cancel_plate_label,
    sample_time_bin_frames,
    save_plate_image,
    save_u10_plate_dataset_yaml,
    save_yolo_plate_label,
    validate_u10_plate_selection,
    yolo_bbox_to_opencv,
)
from snatch_analysis.video import iter_video_frames, read_video_frame


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요.")
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

## 1. 설정

처음에는 모든 dictionary를 비우고 실행 flag를 `False`로 둡니다. 후보 contact sheet를 본 뒤 선택한 영상만 기록합니다. **54개 전체의 원판 유무를 먼저 라벨링하지 않습니다.**

In [ ]:
U10_HAS_PLATE_LABELS: dict[str, str] = {
    "a04_sn_f_01": "yes",
    "a05_sn_f_01": "yes",
    "b03_sn_f_01": "yes",
    "c02_sn_f_01": "yes",
    "a07_sn_f_01": "yes",
    "c08_sn_f_01": "yes",
}
U10_VIDEO_ROLES: dict[str, str] = {
    "a04_sn_f_01": "train",
    "a05_sn_f_01": "train",
    "b03_sn_f_01": "train",
    "c02_sn_f_01": "train",
    "a07_sn_f_01": "validation",
    "c08_sn_f_01": "validation",
}
U10_SELECTION_REASONS: dict[str, str] = {
    "a04_sn_f_01": "큰 초록 원판과 주변 기구가 함께 있는 복잡한 배경",
    "a05_sn_f_01": "카메라에 가까운 원판과 다른 촬영 거리",
    "b03_sn_f_01": "검은 원판과 밝은 벽 배경",
    "c02_sn_f_01": "다른 체형과 검은 원판 크기",
    "a07_sn_f_01": "검증용 초록 원판과 후방 사람·기구",
    "c08_sn_f_01": "검증용 새로운 사람과 카메라 거리",
}

SAVE_U10_METADATA = False
RUN_U10_EXTRACTION = False
RUN_U10_LABELING = False
OVERWRITE_U10_IMAGES = False
OVERWRITE_U10_LABELS = False

SAMPLES_PER_VIDEO = 20
MANIFEST_PATH = PROJECT_ROOT / "data/metadata/videos.csv"
BASELINE_DATASET_ROOT = PROJECT_ROOT / "data/plate_dataset"
DATASET_ROOT = PROJECT_ROOT / "data/plate_dataset_v2"
IMAGES_ROOT = DATASET_ROOT / "images"
LABELS_ROOT = DATASET_ROOT / "labels"
VIDEO_SPLIT_PATH = DATASET_ROOT / "video_split.csv"
SAMPLE_MANIFEST_PATH = DATASET_ROOT / "sample_frames.csv"
DATASET_YAML_PATH = DATASET_ROOT / "dataset.yaml"
U10_ARTIFACT_ROOT = PROJECT_ROOT / "artifacts/u10_plate_dataset_v2"

if SAMPLES_PER_VIDEO != 20:
    raise ValueError("U10 비교에서는 영상당 정확히 20프레임을 사용합니다.")
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest가 없습니다: {MANIFEST_PATH}")

## 2. 후보 영상 확인

평가 역할과 B00을 제외하고 `training_development` 사람마다 한 영상을 보여줍니다. 이 화면에서 원판 유무, 원판의 화면 내 크기, 배경과 카메라 거리를 확인합니다. A/B/C 균형보다 시각적 다양성이 우선입니다.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
eligible = manifest.loc[
    manifest["split_role"].eq("training_development")
    & manifest["read_status"].eq("ok")
    & manifest["lift"].eq("sn")
    & manifest["video_id"].ne(B00_VIDEO_ID)
    & manifest["has_plate"].ne("no")
].copy()
eligible["completion_order"] = eligible["completion"].map({"f": 0, "h": 1})
candidate_review = (
    eligible.sort_values(["person_id", "completion_order", "rep_index"])
    .groupby("person_id", as_index=False)
    .head(1)
    .sort_values(["group", "person_id"])
    .reset_index(drop=True)
)
if candidate_review.empty:
    raise ValueError("확인할 training_development 후보가 없습니다.")

column_count = 4
row_count = math.ceil(len(candidate_review) / column_count)
figure, axes = plt.subplots(row_count, column_count, figsize=(16, 4 * row_count))
axes = np.atleast_1d(axes).ravel()
for axis, row in zip(axes, candidate_review.itertuples(index=False)):
    frame_index = int(row.frame_count) // 2
    frame = read_video_frame(PROJECT_ROOT / row.path, frame_index)
    axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axis.set_title(
        f"{row.video_id} | person={row.person_id} | has_plate={row.has_plate}"
    )
    axis.axis("off")
for axis in axes[len(candidate_review):]:
    axis.axis("off")
plt.tight_layout()
plt.show()
display(
    candidate_review[[
        "video_id", "group", "person_id", "completion",
        "frame_count", "has_plate", "path",
    ]]
)

## 3. 선택 후보의 원판 유무 기록

contact sheet에서 직접 확인한 후보만 `U10_HAS_PLATE_LABELS`에 `yes` 또는 `no`로 입력합니다. 저장할 때만 `SAVE_U10_METADATA=True`로 바꾸고, 저장 후 다시 `False`로 되돌립니다.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
candidate_ids = set(candidate_review["video_id"].astype(str))
unknown_label_ids = sorted(set(U10_HAS_PLATE_LABELS) - candidate_ids)
if unknown_label_ids:
    raise ValueError("후보 contact sheet에 없는 video_id: " + ", ".join(unknown_label_ids))
if SAVE_U10_METADATA:
    if not U10_HAS_PLATE_LABELS:
        raise ValueError("저장할 U10_HAS_PLATE_LABELS가 비어 있습니다.")
    manifest = set_has_plate_labels(manifest, U10_HAS_PLATE_LABELS)
    save_video_manifest(manifest, MANIFEST_PATH)
    print(f"has_plate 저장 완료: {len(U10_HAS_PLATE_LABELS)}개 영상")
else:
    print("manifest 미저장: 확인 후 SAVE_U10_METADATA=True로 한 번 실행하세요.")
display(
    manifest.loc[
        manifest["video_id"].isin(U10_HAS_PLATE_LABELS),
        ["video_id", "person_id", "has_plate"],
    ]
)

## 4. 신규 6개 영상 역할 확정

`U10_VIDEO_ROLES`에 `train` 4개와 `validation` 2개를 입력합니다. 같은 사람은 두 역할로 나눌 수 없습니다. `U10_SELECTION_REASONS`에는 각 영상이 추가하는 원판 크기·색·배경·가림 차이를 한 줄로 기록합니다.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
if not U10_VIDEO_ROLES and not U10_SELECTION_REASONS:
    selected_videos = pd.DataFrame()
    print("U10 선택 대기: 후보 확인 후 두 dictionary에 6개 영상을 입력하세요.")
else:
    if set(U10_VIDEO_ROLES) != set(U10_SELECTION_REASONS):
        raise ValueError("U10_VIDEO_ROLES와 U10_SELECTION_REASONS의 video_id가 다릅니다.")
    manifest_by_video = manifest.set_index("video_id", drop=False)
    unknown_ids = sorted(set(U10_VIDEO_ROLES) - set(manifest_by_video.index.astype(str)))
    if unknown_ids:
        raise ValueError("Manifest에 없는 video_id: " + ", ".join(unknown_ids))
    selection = pd.DataFrame(
        [
            {
                "video_id": video_id,
                "person_id": str(manifest_by_video.loc[video_id, "person_id"]),
                "dataset_role": role,
                "selection_reason": U10_SELECTION_REASONS[video_id],
            }
            for video_id, role in U10_VIDEO_ROLES.items()
        ],
        columns=("video_id", "person_id", "dataset_role", "selection_reason"),
    )
    selection = validate_u10_plate_selection(
        selection, manifest, baseline_video_id=B00_VIDEO_ID
    )
    selected_videos = selection.merge(
        manifest[[
            "video_id", "group", "path", "fps", "frame_count",
            "width", "height", "completion", "has_plate",
        ]],
        on="video_id",
        how="left",
        validate="one_to_one",
    )
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    selected_videos.to_csv(VIDEO_SPLIT_PATH, index=False)
    display(selected_videos)

## 5. 시간 구간별 120프레임 표본

정확한 phase event를 새로 입력하지 않습니다. 각 영상을 시간상 다섯 구간으로 나누고 구간마다 4프레임을 선택합니다. 이 `sampling_bin`은 detector 데이터 다양성을 위한 구간이며 동작 phase 라벨이 아닙니다.

In [ ]:
if not VIDEO_SPLIT_PATH.exists():
    sample_frames = pd.DataFrame()
    print("sample 생성 대기: 먼저 6개 영상 selection을 저장하세요.")
else:
    selected_videos = pd.read_csv(VIDEO_SPLIT_PATH)
    sample_tables = [
        sample_time_bin_frames(
            row.video_id,
            int(row.frame_count),
            sample_count=SAMPLES_PER_VIDEO,
        ).assign(dataset_role=row.dataset_role)
        for row in selected_videos.itertuples(index=False)
    ]
    sample_frames = pd.concat(sample_tables, ignore_index=True)
    if len(sample_frames) != 120:
        raise RuntimeError(f"U10 sample은 120개여야 합니다: {len(sample_frames)}")
    if sample_frames.duplicated(["video_id", "frame_index"]).any():
        raise RuntimeError("U10 sample에 중복 video/frame이 있습니다.")
    sample_frames["image_filename"] = sample_frames.apply(
        lambda row: f"{row.video_id}_frame_{int(row.frame_index):04d}.jpg",
        axis=1,
    )
    sample_frames["label_filename"] = sample_frames["image_filename"].str.replace(
        ".jpg", ".txt", regex=False
    )
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    sample_frames.to_csv(SAMPLE_MANIFEST_PATH, index=False)
    display(
        sample_frames.groupby(["dataset_role", "video_id", "sampling_bin"])
        .size()
        .rename("sample_count")
        .reset_index()
    )

## 6. 이미지 추출

`RUN_U10_EXTRACTION=True`일 때만 파일을 만듭니다. 신규 영상은 한 번씩 순차 디코딩하고 선택 frame만 저장합니다. 기존 B00 30개 이미지·라벨은 원본을 건드리지 않고 V2 train에 복사합니다.

In [ ]:
extraction_actions = []
if not RUN_U10_EXTRACTION:
    print("이미지 미생성: sample 표를 확인한 뒤 RUN_U10_EXTRACTION=True로 바꾸세요.")
else:
    if not VIDEO_SPLIT_PATH.exists() or not SAMPLE_MANIFEST_PATH.exists():
        raise FileNotFoundError("video_split.csv와 sample_frames.csv가 필요합니다.")
    selected_videos = pd.read_csv(VIDEO_SPLIT_PATH)
    sample_frames = pd.read_csv(SAMPLE_MANIFEST_PATH)
    for role in ("train", "validation"):
        (IMAGES_ROOT / role).mkdir(parents=True, exist_ok=True)
        (LABELS_ROOT / role).mkdir(parents=True, exist_ok=True)

    baseline_images = sorted((BASELINE_DATASET_ROOT / "images").glob("*.jpg"))
    baseline_labels = sorted((BASELINE_DATASET_ROOT / "labels").glob("*.txt"))
    if len(baseline_images) != 30 or len(baseline_labels) != 30:
        raise RuntimeError("B00 baseline 이미지와 라벨은 각각 30개여야 합니다.")
    for source_path in [*baseline_images, *baseline_labels]:
        destination_root = IMAGES_ROOT if source_path.suffix == ".jpg" else LABELS_ROOT
        destination_path = destination_root / "train" / source_path.name
        if destination_path.exists():
            action = "skipped"
        else:
            shutil.copy2(source_path, destination_path)
            action = "created"
        extraction_actions.append({"source": str(source_path), "action": action})

    metadata_by_video = selected_videos.set_index("video_id")
    for video_id, video_samples in sample_frames.groupby("video_id"):
        metadata = metadata_by_video.loc[video_id]
        targets = video_samples.set_index("frame_index")
        saved_indices = set()
        for frame_index, _, frame in iter_video_frames(
            PROJECT_ROOT / metadata.path,
            expected_frame_count=int(metadata.frame_count),
        ):
            if frame_index not in targets.index:
                continue
            sample = targets.loc[frame_index]
            image_path = IMAGES_ROOT / sample.dataset_role / sample.image_filename
            action = save_plate_image(
                image_path, frame, overwrite=OVERWRITE_U10_IMAGES
            )
            extraction_actions.append(
                {"video_id": video_id, "frame_index": frame_index, "action": action}
            )
            saved_indices.add(frame_index)
        missing_indices = sorted(set(targets.index.astype(int)) - saved_indices)
        if missing_indices:
            raise RuntimeError(f"{video_id} 추출 누락 frame: {missing_indices}")
    display(pd.DataFrame(extraction_actions)["action"].value_counts().rename("count"))

## 7. 신규 120프레임 수동 bbox

`RUN_U10_LABELING=True`로 바꾸면 아직 라벨이 없는 이미지부터 OpenCV ROI 창이 열립니다. 주동작자가 드는 카메라 쪽 원판 하나만 지정합니다. 취소한 frame은 좌표를 만들지 않고 `plate_label_cancelled`로 표에 남습니다. 재라벨링(`OVERWRITE_U10_LABELS=True`) 중 취소하면 잘못된 기존 bbox를 삭제합니다.

In [ ]:
label_actions = []
if not RUN_U10_LABELING:
    print("라벨링 미실행: 이미지 검토 후 RUN_U10_LABELING=True로 바꾸세요.")
else:
    if not SAMPLE_MANIFEST_PATH.exists():
        raise FileNotFoundError("sample_frames.csv가 필요합니다.")
    sample_frames = pd.read_csv(SAMPLE_MANIFEST_PATH)
    try:
        for row in sample_frames.itertuples(index=False):
            image_path = IMAGES_ROOT / row.dataset_role / row.image_filename
            label_path = LABELS_ROOT / row.dataset_role / row.label_filename
            if label_path.exists() and not OVERWRITE_U10_LABELS:
                label_actions.append(
                    {"video_id": row.video_id, "frame_index": row.frame_index, "action": "skipped"}
                )
                continue
            frame = cv2.imread(str(image_path))
            if frame is None:
                raise FileNotFoundError(f"추출 이미지가 없습니다: {image_path}")
            bbox = cv2.selectROI(
                f"plate | {row.video_id} | frame={int(row.frame_index)}",
                frame,
                showCrosshair=True,
                fromCenter=False,
            )
            if bbox[2] <= 0 or bbox[3] <= 0:
                action = cancel_plate_label(
                    label_path, overwrite=OVERWRITE_U10_LABELS
                )
            else:
                action = save_yolo_plate_label(
                    label_path,
                    tuple(float(value) for value in bbox),
                    image_width=frame.shape[1],
                    image_height=frame.shape[0],
                    overwrite=OVERWRITE_U10_LABELS,
                )
            label_actions.append(
                {"video_id": row.video_id, "frame_index": row.frame_index, "action": action}
            )
    finally:
        cv2.destroyAllWindows()
    display(pd.DataFrame(label_actions))

## 8. bbox contact sheet 검수

신규 120개 라벨을 20개씩 나눠 contact sheet로 저장합니다. 초록 bbox가 주동작 원판 하나를 가리키는지 모두 확인합니다. B00 30개는 U6에서 이미 검수했으므로 이 화면에서는 신규 데이터만 다시 봅니다.

In [ ]:
contact_sheet_paths = []
if not SAMPLE_MANIFEST_PATH.exists():
    print("contact sheet 대기: sample_frames.csv가 없습니다.")
else:
    sample_frames = pd.read_csv(SAMPLE_MANIFEST_PATH)
    if len(sample_frames) != 120:
        raise RuntimeError(f"신규 U10 sample은 120개여야 합니다: {len(sample_frames)}")
    review_entries = []
    missing_labels = []
    for row in sample_frames.itertuples(index=False):
        image_path = IMAGES_ROOT / row.dataset_role / row.image_filename
        label_path = LABELS_ROOT / row.dataset_role / row.label_filename
        if not image_path.exists() or not label_path.exists():
            missing_labels.append((row.video_id, int(row.frame_index)))
            continue
        frame = cv2.imread(str(image_path))
        if frame is None:
            raise OSError(f"이미지를 읽을 수 없습니다: {image_path}")
        tokens = label_path.read_text(encoding="utf-8").strip().split()
        if len(tokens) != 5 or tokens[0] != "0":
            raise ValueError(f"잘못된 YOLO 원판 라벨: {label_path}")
        bbox = yolo_bbox_to_opencv(
            tuple(float(value) for value in tokens[1:]),
            image_width=frame.shape[1],
            image_height=frame.shape[0],
        )
        review_entries.append((row, frame, bbox))
    print(f"신규 라벨: {len(review_entries)}/120, 미완료: {len(missing_labels)}")
    if missing_labels:
        display(pd.DataFrame(missing_labels, columns=["video_id", "frame_index"]))
    else:
        U10_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
        page_size = 20
        column_count = 5
        for page_index, start in enumerate(range(0, len(review_entries), page_size), start=1):
            page_entries = review_entries[start:start + page_size]
            row_count = math.ceil(len(page_entries) / column_count)
            figure, axes = plt.subplots(row_count, column_count, figsize=(16, 3.5 * row_count))
            axes = np.atleast_1d(axes).ravel()
            for axis, (row, frame, bbox) in zip(axes, page_entries):
                x, y, width, height = bbox
                marked = frame.copy()
                cv2.rectangle(marked, (x, y), (x + width, y + height), (0, 255, 0), 3)
                axis.imshow(cv2.cvtColor(marked, cv2.COLOR_BGR2RGB))
                axis.set_title(f"{row.video_id} | frame={int(row.frame_index)}")
                axis.axis("off")
            for axis in axes[len(page_entries):]:
                axis.axis("off")
            plt.tight_layout()
            output_path = U10_ARTIFACT_ROOT / f"label_contact_sheet_{page_index:02d}.png"
            figure.savefig(output_path, dpi=120, bbox_inches="tight")
            contact_sheet_paths.append(output_path)
            plt.show()
            plt.close(figure)
        display(pd.DataFrame({"contact_sheet_path": contact_sheet_paths}))

## 9. 학습용 dataset YAML

신규 120개와 B00 30개가 모두 준비됐을 때만 YAML을 씁니다. train은 B00 30개와 신규 train 80개, validation은 학습에 포함되지 않은 두 사람의 40개입니다. 현재 split의 파일명과 정확히 일치하지 않으면 이전 YAML도 무효화합니다.

In [ ]:
if not VIDEO_SPLIT_PATH.exists() or not SAMPLE_MANIFEST_PATH.exists():
    DATASET_YAML_PATH.unlink(missing_ok=True)
    print("dataset.yaml 대기: video_split.csv와 sample_frames.csv가 필요합니다.")
else:
    selected_videos = pd.read_csv(VIDEO_SPLIT_PATH)
    sample_frames = pd.read_csv(SAMPLE_MANIFEST_PATH)
    dataset_counts = save_u10_plate_dataset_yaml(
        DATASET_ROOT,
        BASELINE_DATASET_ROOT,
        selected_videos,
        sample_frames,
    )
    print(f"dataset YAML 저장: {DATASET_YAML_PATH}")
    display(pd.DataFrame([dataset_counts]))

## 다음 checkpoint

여섯 영상 선택, 신규 120개 bbox와 contact sheet 검수를 마친 뒤 학습으로 넘어갑니다. 이 단계에서는 모델 설정·confidence·tracker를 바꾸지 않습니다.